# Session 5: Multi-Agent Orchestration and Subagent Configuration

**Task statements:** 1.2 (coordinator-subagent orchestration) and 1.3 (subagent invocation, context passing, spawning)
**Domains:** D1 Agentic Architecture (27%), with D2 and D5 underneath
**Exercise:** Exercise 4, *Multi-agent research pipeline*, starts today (continues in Session 14). It reuses the agent loop from Exercise 1 and the error contract from Session 4.
**Pre-work:** Agent SDK docs on subagents. The API course's parallelization and routing lessons are useful background.

| Time | Section | What you do |
|---|---|---|
| 0 to 5 | 1. Setup and Session 4 recap | Copy the project forward, answer 4 recap questions |
| 5 to 9 | 2. Hub and spoke in 4 minutes | Read the pattern, predict what a subagent can see |
| 9 to 14 | 3. Exercise A | Design the Scenario 3 system before you build it |
| 14 to 32 | 4. Build | TODO 1 to 4: prompt, isolated subagent, delegate tool, parallel fan-out |
| 32 to 39 | 5. Run it | TODO 5 coordinator prompt, live run, parallel vs sequential latency |
| 39 to 46 | 6. Exercise B | Diagnose an incomplete report (sample Q7) |
| 46 to 53 | 7. Agent SDK | TODO 6 AgentDefinitions, `allowed_tools`, dry run |
| 53 to 60 | 8. Exam practice | Sample Q9 + 5 scenario questions, homework |

**Research question for the whole session**

> *Why did return rates at Northwind Outfitters rise in Q3 2026, and what should we change before the holiday season?*

The evidence is spread across three kinds of source (public web articles, internal documents, the returns database) and points to five different drivers. No single agent with one toolset can find all of it. That is the reason we need a coordinator.

## 1. Setup and Session 4 recap (5 min)

The cell below brings the `northwind` project forward from Session 4 **without overwriting** the new Session 5 files, finds your `.env`, and picks live or simulated mode.

* `USE_LIVE_API = True` uses Claude (Haiku 5.5, a few cents for the whole session).
* `USE_LIVE_API = False` uses a scripted, thread-safe fake client. Everything still runs, including timings.

In [ ]:
import os, sys, shutil, json, time, glob
from pathlib import Path

HERE = Path.cwd()
NW = HERE / "northwind"
PREV = sorted(glob.glob(str(HERE.parent / "Session 04*" / "northwind")))

# 1) copy Session 4 files forward, never overwriting Session 5 files
if PREV:
    src = Path(PREV[0])
    copied = 0
    for f in src.rglob("*"):
        if "__pycache__" in f.parts or f.is_dir():
            continue
        dest = NW / f.relative_to(src)
        if not dest.exists():
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied += 1
    print(f"Copied {copied} file(s) forward from {src.parent.name}")
else:
    print("Session 04 folder not found next to this one. Session 5 files run on their own; continue.")

# 2) API key: environment first, then a .env in this or an earlier session folder
for env_file in [NW / ".env", HERE / ".env", *sorted(HERE.parent.glob("Session 0*/.env")), *sorted(HERE.parent.glob("Session 0*/northwind/.env"))]:
    if os.environ.get("ANTHROPIC_API_KEY"):
        break
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.strip().startswith("ANTHROPIC_API_KEY="):
                os.environ["ANTHROPIC_API_KEY"] = line.split("=", 1)[1].strip().strip('"').strip("'")
                print("Loaded ANTHROPIC_API_KEY from", env_file)

os.chdir(NW)
sys.path.insert(0, str(NW))

USE_LIVE_API = bool(os.environ.get("ANTHROPIC_API_KEY"))   # set False to force the simulated client
print("Mode:", "LIVE (Claude Haiku 5.5)" if USE_LIVE_API else "SIMULATED (scripted fake client)")

In [ ]:
from research_corpus import WEB_SOURCES, INTERNAL_DOCS, RETURN_STATS
from research_tools import RESEARCH_TOOL_DEFS, run_research_tool, inject_research_fault, clear_research_faults
from subagents import SUBAGENTS, SubagentSpec, REPORT_CONTRACT
from orchestration import (RESEARCH_QUESTION, NARROW_COORDINATOR_SYSTEM, run_coordinator,
                           print_trace, parse_report, failed_report)
from fake_multi_client import RoutingFakeClient, good_scripts, narrow_scripts, fault_scripts
from answer_keys_s5 import coverage_report, review_design, DIAGNOSIS_KEY, DIAGNOSIS_EXPLANATION
import test_multi_agent_offline as checks


def make_client(scenario="good", latency=0.25):
    # Live: the real client. Simulated: a fake that follows the named script.
    if USE_LIVE_API:
        import anthropic
        return anthropic.Anthropic()
    return RoutingFakeClient({"good": good_scripts, "narrow": narrow_scripts, "fault": fault_scripts}[scenario](),
                             latency=latency)

print("Ready. Research sources:", len(WEB_SOURCES), "web,", len(INTERNAL_DOCS), "internal docs, 1 dataset")

### Session 4 recap: answer out loud, then check

1. A tool returns `{"orders": []}` for a customer with no orders. Is that an error?
2. `get_order_history` times out once. Who should retry: the subagent that called it, or the coordinator?
3. The retry also fails. What must the subagent send upward besides "failed"?
4. A refund over the $500 limit comes back as a business error. Should the agent retry it?

<details><summary>Answers</summary>

1. No. A valid empty result is a success. Marking it as an error (or an access failure as "empty") is the anti-pattern from 5.3.
2. The subagent, locally. Transient errors are recovered close to where they happen.
3. Failure type, what was attempted, partial results, and alternatives (sample Q8). That is the shape `failed_report()` builds today.
4. No. `isRetryable: false`. Explain the policy to the customer, offer escalation. Session 6 adds the hook that blocks it before the call.
</details>

Today the Session 4 contract stops being one tool's concern and becomes the contract **between agents**: every subagent report carries `status` (complete, partial, failed), `coverage`, and `gaps`.

## 2. Hub and spoke in 4 minutes

```
                       user question
                            |
                     +-------------+
                     | COORDINATOR |   tools: delegate_task only
                     +-------------+   decompose -> delegate -> aggregate -> check gaps -> (re-delegate)
            task+context /    |    \ task+context        (several calls in ONE response = parallel)
                        v     v     v
            +--------------+ +-------------+ +--------------+
            |web_researcher| | doc_analyst | | data_analyst |   each: own system prompt,
            | search_web   | | search_docs | | query_return |         own scoped tools,
            | fetch_page   | | read_doc    | |   _stats     |         EMPTY history
            +--------------+ +-------------+ +--------------+
                        \     |     /
                  structured reports (findings + source metadata + status + gaps)
```

Four rules that the exam tests again and again:

| Rule | Why |
|---|---|
| **All traffic goes through the coordinator** (hub and spoke) | One place to see, check, and log what moved between agents |
| **A subagent inherits nothing.** Its context = its system prompt + the task prompt you write | So the coordinator must pass facts, earlier findings and constraints *explicitly* |
| **Parallel = several spawn calls in one coordinator response** | One call per turn means sequential, whatever the code can do |
| **If every subagent succeeded and the report is still incomplete, look at the decomposition** | Subagents only answer what they were asked (sample Q7) |

**Predict before you build:** the coordinator has read a QA report saying pole batch TP-0726 cracked. It now asks `data_analyst` to "check whether the defect shows up in the numbers". What does `data_analyst` know about TP-0726? (Answer: nothing, unless the coordinator wrote it into the task prompt.)

In [ ]:
# A quick look at the specialists and what each one is allowed to touch
for name, spec in SUBAGENTS.items():
    print(f"{name:<15} tools={list(spec.tools)}")
    print(f"{'':<15} when: {spec.description[:95]}...")

## 3. Exercise A: design the Scenario 3 system (5 min)

Scenario 3 in the exam guide is a multi-agent research system. Design ours **before** looking at the code. Fill in `MY_DESIGN` in your own words; short phrases are fine. Then run the review cell and compare with the key.

Think about: how to split the work (by product? by source type? by driver?), which tools each agent gets, what the coordinator must pass to each agent, what each agent sends back, and what can run in parallel.

In [ ]:
MY_DESIGN = {
    "subagents": "",               # which specialists, split by what?
    "tools_per_agent": "",         # who gets which tools; what does the coordinator get?
    "context_passed": "",          # what goes into each task prompt?
    "output_format": "",           # what does a subagent return?
    "parallel_or_sequential": "",  # what runs together, what has to wait?
    "refinement": "",              # what happens when the synthesis has a gap?
}
review_design(MY_DESIGN)

**Instructor discussion (1 min):** why split by *source type* rather than by *product category*? Because the drivers of returns cut across products (shipping damage hits every category), but every source type needs different tools. Splitting by product would give three agents the same tools and the same blind spots. Keep this answer in mind for section 6.

## 4. Build the pipeline (18 min)

You write four functions. Each has a check cell right after it. The coordinator loop itself is provided in `orchestration.run_coordinator`: it is your Session 2 loop, except the coordinator's only tool is `delegate_task` and all delegate calls from one response go to `execute_delegations` together.

### TODO 1: the task prompt is the subagent's whole world

A subagent starts with an **empty history**. The only bridge from the coordinator is the string you build here. It must contain:

1. the task,
2. the context the coordinator chose to pass (question, period, earlier findings *with their source ids*),
3. a scope note (it cannot see anything else),
4. the output contract `REPORT_CONTRACT`, so findings come back with their metadata kept separate.

In [ ]:
print(REPORT_CONTRACT)

In [ ]:
# TODO 1
def build_subagent_prompt(task: str, context: str, contract: str = REPORT_CONTRACT) -> str:
    # Return ONE string with four labelled parts: task, context, scope, output format.
    # Keep the context verbatim: source ids like NW-QA-2026-031 must survive.
    raise NotImplementedError("TODO 1")

In [ ]:
# Check TODO 1
try:
    ok, what = checks.check_prompt(globals()); print("PASS" if ok else "FAIL", what)
    print("\n--- example ---\n" + build_subagent_prompt("Measure Q3 tent returns.", "TP-0726 cracked (NW-QA-2026-031)")[:400])
except NotImplementedError as e:
    print("Not implemented yet:", e)

### TODO 2: run one subagent in isolation

This is your Session 2 `run_agent` with three changes that make it a *subagent*:

| Change | Line of code | Task statement link |
|---|---|---|
| Fresh history | `messages = [{"role": "user", "content": prompt}]` | 1.3 subagents inherit nothing |
| Its own system prompt and **only its own tools** | `system=spec.system_prompt`, `tools=[RESEARCH_TOOL_DEFS[n] for n in spec.tools]` | 2.3 least privilege, carried into 1.3 |
| Returns a **structured report**, never raises on bad output | `parse_report(text)`, else `failed_report(...)` | 5.3 propagation |

Also refuse any tool call outside `spec.tools` with a `permission` error (defence in depth: the model should never see those tools, but code that trusts the model is the anti-pattern).

Use `run_research_tool(name, input)` to execute tools; it never raises and returns the Session 4 error shape on failure. Mark `"is_error": True` on the tool_result when the output has `isError`.

In [ ]:
# TODO 2
def run_subagent(spec, prompt: str, client, verbose: bool = False) -> dict:
    tools = None      # TODO: only this spec's tools, from RESEARCH_TOOL_DEFS
    messages = None   # TODO: a fresh history containing just the prompt

    for _ in range(spec.max_iterations):
        # TODO: call client.messages.create(model=spec.model, max_tokens=2048, system=..., tools=..., messages=...)
        # TODO: append the assistant turn
        # TODO: if stop_reason == "tool_use": run each tool_use block (refuse out-of-scope names with a
        #       permission error), append ONE user turn with all tool_result blocks, continue
        # TODO: otherwise join the text blocks, parse_report(text); on None return
        #       failed_report(spec.name, "output_format", "...", raw=text); else return the report
        raise NotImplementedError("TODO 2")

    return failed_report(spec.name, "max_iterations", f"No report after {spec.max_iterations} turns.")

In [ ]:
# Check TODO 2 (scripted, no API calls)
for chk in [checks.check_isolation, checks.check_tool_loop, checks.check_out_of_scope, checks.check_bad_output]:
    try:
        ok, what = chk(globals()); print("PASS" if ok else "FAIL", f"{chk.__name__:<20}", what)
    except NotImplementedError as e:
        print("Not implemented yet:", e); break
    except Exception as e:
        print("FAIL", f"{chk.__name__:<20}", type(e).__name__, e)
# Instructor rescue, only if stuck: from solutions.multi_agent_ref import run_subagent

In [ ]:
# Try one subagent on its own (live if you have a key). Watch: it knows only what the prompt says.
if USE_LIVE_API:
    import anthropic
    rep = run_subagent(SUBAGENTS["data_analyst"],
                       build_subagent_prompt("Measure the change from Q2 to Q3 2026 in return rate by product category.",
                                             "Northwind Outfitters, online camping gear. Compare Q3 2026 with Q2 2026."),
                       anthropic.Anthropic(), verbose=True)
    print(json.dumps(rep, indent=2)[:1500])
else:
    print("Simulated mode: skip. The end-to-end run in section 5 exercises run_subagent.")

**Ask the learner:** the report says `"status": "partial"`. Why would a correct data_analyst say partial when every query succeeded? (Coverage: Q3 data only runs to 2026-09-21. A complete-looking number that hides a partial period is exactly what 5.3 calls an empty-success anti-pattern in disguise.)

### TODO 3: the delegate tool (our `Task`)

In the Agent SDK the coordinator spawns subagents with the built-in **Agent** tool (the exam guide calls it **Task**, its name before Claude Code v2.1.63). With the raw API we define it ourselves. Build it **from the registry** so a new subagent becomes selectable without touching the coordinator: that is dynamic subagent selection, driven by descriptions (the same lesson as Session 3's tool descriptions).

Requirements: name `delegate_task`; `subagent_type` is an enum of the registry keys; `task` and `context` both **required**; the description includes the roster (each agent's description) and says plainly that subagents start with no memory and that independent tasks should be sent in the same response.

In [ ]:
# TODO 3
def make_delegate_tool(subagents=SUBAGENTS) -> dict:
    raise NotImplementedError("TODO 3")

In [ ]:
# Check TODO 3
try:
    ok, what = checks.check_delegate_tool(globals()); print("PASS" if ok else "FAIL", what)
    print(json.dumps(make_delegate_tool(), indent=1)[:900])
except NotImplementedError as e:
    print("Not implemented yet:", e)

**Why is `context` required, not optional?** Because an optional field gets skipped. When it is required, the model has to decide what to pass every single time, and an empty context is a visible choice in the log instead of a silent omission.

### TODO 4: fan out, keep order, contain failures

`execute_delegations(tool_uses, client, parallel=True)` receives **every** `delegate_task` block from one coordinator response.

* Build the prompt with TODO 1, run the subagent with TODO 2.
* `parallel=True` and more than one call: run them concurrently (`ThreadPoolExecutor`; `pool.map` keeps the order).
* Return **one** `tool_result` per tool_use, same order, `content=json.dumps(report)`, `is_error=True` when the report's status is `failed`.
* Unknown `subagent_type` or a subagent that raises: turn it into a `failed_report(...)`. Never let one subagent kill the others (5.3: do not terminate the whole workflow on one failure).

In [ ]:
# TODO 4
from concurrent.futures import ThreadPoolExecutor

def execute_delegations(tool_uses, client, parallel: bool = True, subagents=SUBAGENTS) -> list:
    raise NotImplementedError("TODO 4")

In [ ]:
# Check TODO 4
for chk in [checks.check_parallel, checks.check_failure_isolated]:
    try:
        ok, what = chk(globals()); print("PASS" if ok else "FAIL", f"{chk.__name__:<22}", what)
    except NotImplementedError as e:
        print("Not implemented yet:", e); break

In [ ]:
# Time check. If a TODO above is still unfinished at minute 32, run this: it loads the reference for
# ONLY the functions that still raise NotImplementedError, so the rest of the session can go on.
# Finish the missing TODO as homework (the save cell at the end keeps the reference until yours passes).
import solutions.multi_agent_ref as _ref
_probe = {
    "build_subagent_prompt": lambda: build_subagent_prompt("t", "c"),
    "run_subagent": lambda: checks.check_isolation(globals()),
    "make_delegate_tool": lambda: make_delegate_tool(),
    "execute_delegations": lambda: checks.check_failure_isolated(globals()),
}
for _name, _call in _probe.items():
    try:
        _call()
    except NotImplementedError:
        globals()[_name] = getattr(_ref, _name)
        print("loaded reference for", _name)
print("ready for section 5")

## 5. Run it (7 min)

### TODO 5: a coordinator prompt with goals and quality criteria, not a procedure

The exam guide's phrasing: coordinator prompts should state **goals and quality criteria**, not step-by-step procedures. A procedure ("call X three times, then combine") freezes your assumptions about the problem into the decomposition. Criteria let the coordinator decompose for the problem in front of it and *check its own coverage* before it writes.

Write `COORDINATOR_SYSTEM` with: the role (plans, delegates, synthesizes; does no research itself), the goal, quality criteria (coverage of the whole customer journey, evidence with source ids and dates, honesty about gaps and conflicts, actionable recommendations), and two working habits (pass everything a subagent needs; send independent tasks together; check coverage and re-delegate before writing).

In [ ]:
# TODO 5
COORDINATOR_SYSTEM = """TODO 5: replace this text with your coordinator system prompt."""

In [ ]:
ok, what = checks.check_coordinator_prompt(globals()); print("PASS" if ok else "FAIL", what)

In [ ]:
# The full run, parallel. Live: about 30 to 90 seconds with Haiku. Simulated: about 5 seconds.
client = make_client("good")
report_text, coord_messages, trace_par = run_coordinator(
    RESEARCH_QUESTION, client, system=COORDINATOR_SYSTEM, delegate_tool=make_delegate_tool(),
    execute_delegations=execute_delegations, parallel=True)
print()
print_trace(trace_par)
print("\n" + report_text)

In [ ]:
cov_par = coverage_report(report_text)

**Read the trace with the learner:**

* Round 1: how many `delegate_task` calls came in *one* response? That number is your parallelism.
* Is there a round 2? If yes, read its task and context: the coordinator found a gap (often returns policy) and passed an earlier finding with its source id into the follow-up. That is **iterative refinement** (check synthesis for gaps, re-delegate, re-synthesize) and **explicit context passing between subagents through the hub**, in one move.
* Which statuses came back? A `partial` from data_analyst should appear in the report's gaps.

### Latency comparison (homework step 2 starts here)

Same question, same code, `parallel=False`. Only the execution strategy changes.

In [ ]:
client = make_client("good")
report_seq, _, trace_seq = run_coordinator(
    RESEARCH_QUESTION, client, system=COORDINATOR_SYSTEM, delegate_tool=make_delegate_tool(),
    execute_delegations=execute_delegations, parallel=False, verbose=False)
print_trace(trace_seq)

r1p, r1s = trace_par["rounds"][0]["seconds"], trace_seq["rounds"][0]["seconds"]
print(f"\nRound 1 (fan-out):  parallel {r1p:.1f}s  vs  sequential {r1s:.1f}s  -> {r1s / max(r1p, 0.01):.1f}x")
print(f"Whole run:          parallel {trace_par['total_seconds']:.1f}s  vs  sequential {trace_seq['total_seconds']:.1f}s")

**Ask the learner:** parallel round 1 is roughly the time of the *slowest* subagent; sequential is the *sum*. Why is the whole-run speed-up smaller than the round-1 speed-up? (The coordinator's own turns and any round-2 follow-up are sequential by nature: a follow-up that needs round-1 findings cannot start earlier.) What would make the parallel gain disappear even with this code? (A coordinator that emits one `delegate_task` per response.)

## 6. Exercise B: diagnose an incomplete report (7 min)

A colleague wrote a different coordinator prompt (`NARROW_COORDINATOR_SYSTEM`). Run it with **your** subagent code, unchanged.

In [ ]:
print(NARROW_COORDINATOR_SYSTEM, "\n")
client = make_client("narrow")
narrow_text, _, trace_narrow = run_coordinator(
    RESEARCH_QUESTION, client, system=NARROW_COORDINATOR_SYSTEM, delegate_tool=make_delegate_tool(),
    execute_delegations=execute_delegations, parallel=True)
print()
print_trace(trace_narrow)
print("\n" + narrow_text + "\n")
coverage_report(narrow_text);

**Diagnosis.** Every subagent returned `complete`. The report covers about one driver of five. What is the most likely root cause?

* **A.** The doc_analyst needs more search calls per task.
* **B.** The coordinator's decomposition only assigned product-quality questions by category, so shipping, sizing, listing and policy were never investigated.
* **C.** The synthesis step dropped findings that the subagents returned.
* **D.** The subagents could not reach the web and dataset sources.

Write your letter and one sentence of evidence from the trace, then run the next cell.

In [ ]:
MY_DIAGNOSIS = ""          # "A" / "B" / "C" / "D"
MY_EVIDENCE = ""           # one sentence from the trace
print("Key:", DIAGNOSIS_KEY, "| yours:", MY_DIAGNOSIS or "(blank)")
print(DIAGNOSIS_EXPLANATION)

**Connect to sample Q7 (paraphrased).** In the exam guide's research-system scenario, a report on a broad topic covers only one sub-area. The logs show the coordinator split the topic into three subtopics that all sit inside that one sub-area, and every subagent did its job well. The answer is the coordinator's decomposition, not the subagents, the search, or the synthesis.

The reasoning rule: **when subagents succeed and the output is still incomplete, look upstream at what they were asked.** Here the narrow prompt *forced* the narrow split: an assumption ("returns are a quality problem") plus a procedure. Your TODO 5 prompt fixes it with criteria (coverage of the whole customer journey) instead of steps.

**Note the second lesson in the narrow run:** three tasks went to the *same* agent with the *same* tools. Partitioning by product category gave three copies of one blind spot. Partitioning by source type (or by driver) gives different capabilities.

### Optional (2 min): a subagent fails mid-run
Session 4's contract, now across agents. The returns database times out; does the pipeline die?

In [ ]:
inject_research_fault("query_return_stats", times=5)
try:
    client = make_client("fault")
    fault_text, _, trace_fault = run_coordinator(
        RESEARCH_QUESTION, client, system=COORDINATOR_SYSTEM, delegate_tool=make_delegate_tool(),
        execute_delegations=execute_delegations, parallel=True, verbose=False)
finally:
    clear_research_faults()
print_trace(trace_fault)
print("\n" + fault_text[-600:])

Look for: data_analyst `failed` but the round still returned three results; the final report says what is *not* verified and why. That is "propagate unresolved errors with context; annotate coverage in the synthesis", not "kill the workflow" and not "pretend it worked".

## 7. The same system on the Agent SDK (7 min)

The exam asks about the Agent SDK's configuration, so map what you built onto it:

| You built (raw API) | Agent SDK |
|---|---|
| `SubagentSpec` in `SUBAGENTS` | `AgentDefinition(description, prompt, tools, model)` in `ClaudeAgentOptions(agents={...})` |
| `delegate_task` tool | built-in **Agent** tool (**Task** in the exam guide and before Claude Code v2.1.63). The coordinator must be allowed to use it: `"Agent"` / `"Task"` in `allowed_tools` |
| `subagent_type` enum | the keys of `agents={...}`; Claude picks by `description` |
| `build_subagent_prompt` | the `prompt` argument Claude writes into each Agent call: still the **only** thing passed from parent to child |
| `spec.tools` filter | `AgentDefinition.tools`. Omit it and the subagent inherits every available tool |
| several `delegate_task` blocks in one response, run in threads | several Agent tool calls in one response; the SDK runs them concurrently |
| research functions | `@tool` functions in an in-process MCP server: names become `mcp__research__<tool>` |

### TODO 6: AgentDefinitions and the coordinator's allowed tools

Write `build_agents()` returning one `AgentDefinition` per entry in `SUBAGENTS`: same description, the system prompt **plus** `REPORT_CONTRACT`, tools mapped to MCP names with `mcp_name()`, `model="haiku"`. Then set `COORDINATOR_ALLOWED_TOOLS`: the spawn tool (list both `"Agent"` and `"Task"` so either SDK version works) plus the five research tools so the subagents can run them without permission prompts.

In [ ]:
from claude_agent_sdk import AgentDefinition
from research_tools import RESEARCH_FUNCTIONS

SERVER = "research"

def mcp_name(tool: str) -> str:
    return f"mcp__{SERVER}__{tool}"

# TODO 6
def build_agents() -> dict:
    raise NotImplementedError("TODO 6")

COORDINATOR_ALLOWED_TOOLS = []   # TODO 6

In [ ]:
try:
    for status, what, note in checks.check_sdk_config(build_agents, COORDINATOR_ALLOWED_TOOLS):
        print(status, what, note)
except NotImplementedError as e:
    print("Not implemented yet:", e)

Now write your config to `sdk_config.py` and dry-run the SDK script (no API call). It prints exactly what the SDK will receive.

In [ ]:
import inspect, subprocess

def _write_sdk_config():
    try:
        rows = checks.check_sdk_config(build_agents, COORDINATOR_ALLOWED_TOOLS)
        ok = all(r[0] == "PASS" for r in rows)
    except Exception:
        ok = False
    if not ok:
        shutil.copy("solutions/sdk_config_ref.py", "sdk_config.py")
        return "reference (your TODO 6 is not passing yet)"
    src = ('"""Your Session 5 TODO 6, written by the notebook."""\n'
           "from claude_agent_sdk import AgentDefinition\n\n"
           "from research_tools import RESEARCH_FUNCTIONS\nfrom subagents import REPORT_CONTRACT, SUBAGENTS\n\n"
           f"SERVER = {SERVER!r}\n\n\n" + inspect.getsource(mcp_name) + "\n\n" + inspect.getsource(build_agents)
           + f"\n\nCOORDINATOR_ALLOWED_TOOLS = {COORDINATOR_ALLOWED_TOOLS!r}\n")
    Path("sdk_config.py").write_text(src, encoding="utf-8")
    return "yours"

print("sdk_config.py written:", _write_sdk_config())
print(subprocess.run([sys.executable, "s5_sdk_research.py", "--dry-run"], capture_output=True, text=True).stdout[:2500])

**Live run (if time allows, otherwise homework).** Run these in a terminal from the `northwind` folder, or uncomment the cell. Each run is capped at $1 by `max_budget_usd`; with Haiku it normally costs a few cents.

```
python s5_sdk_research.py               # parallel
python s5_sdk_research.py --sequential  # one subagent at a time
python s5_sdk_research.py --narrow      # the Exercise B prompt on the SDK
```

Watch the log for `SPAWN` lines arriving together (parallel), `(inside subagent)` tool calls (they carry `parent_tool_use_id`), and the summary line with `max concurrent`. A `WARNING: coordinator researched itself` line means the coordinator skipped delegation: tighten its prompt (or, the stronger fix, do not give it the tools).

Configuration details in the script worth pointing at: `disallowed_tools` removes the built-in file and web tools so research stays on our sources; `CLAUDE_AGENT_SDK_DISABLE_BUILTIN_AGENTS=1` leaves only our three subagent types; `CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH=1` stops subagents from spawning their own.

**Fork-based sessions (1.3), one minute.** A normal subagent starts empty. A *fork* starts from a **copy** of the current conversation, so it can explore an alternative without re-sending context and without polluting the original. In the SDK, resume a session with `fork_session=True` to branch it (`ClaudeAgentOptions(resume=session_id, fork_session=True)`); Claude Code can also spawn a fork subagent (`subagent_type: "fork"`). Use a fork when the branch genuinely needs the shared history (compare two approaches after a long investigation); use a fresh subagent with an explicit prompt when it does not (most delegation). Session 10 uses forks in practice.

In [ ]:
# Uncomment to run the SDK version live from here (about 1 to 2 minutes):
# print(subprocess.run([sys.executable, "s5_sdk_research.py"], capture_output=True, text=True, timeout=600).stdout[-4000:])

## 8. Exam practice (7 min)

### Sample Q9 (paraphrased from the exam guide)
In the research system, the synthesis agent often needs to check a simple fact while combining results. Every check goes back through the coordinator to the web search agent, which adds two or three round trips. Most of these checks (about 85%) are simple lookups; a few need deeper investigation. What is the best change?

The guide's answer: give the synthesis agent a **narrowly scoped verification tool** for simple lookups, and keep sending the complex verifications through the coordinator. Why not give it all the web tools? Least privilege and role drift (Session 3). Why not batch every check to the end? It breaks the cases where synthesis depends on the answer. Why not let the synthesis agent call the web agent directly? It breaks hub-and-spoke observability for the cases that need it.

### Five new scenario questions
Answer each, then open the explanation. Ask "why are the other three wrong?" every time.

**Q1 (1.3).** A coordinator sends a synthesis subagent the instruction "Write the final report using the findings so far." The report has no citations and includes figures nobody found. Every research subagent completed normally. What is the most effective fix?
A. Run the synthesis subagent on a larger model.  B. Give the synthesis subagent the search tools so it can find its own figures.  C. Pass the findings, with their source metadata, explicitly in the synthesis subagent's task prompt in a structured format.  D. Turn on the SDK option that lets subagents inherit the coordinator's conversation.

<details><summary>Answer</summary>

**C.** Subagents inherit nothing; "the findings so far" pointed at context the subagent never had, so it invented. A: a bigger model cannot cite text it never received. B: duplicates research and widens its role. D: there is no such inheritance switch for normal subagents (a fork copies the conversation, but that is a different mechanism with a different purpose, and the question asks for the targeted fix): a plausible-sounding feature is the classic distractor.
</details>

**Q2 (1.3).** A coordinator has four independent research subagents. Total runtime equals the sum of the four. Logs show exactly one Task call per coordinator turn. What should you change?
A. Increase `max_turns`.  B. Have the coordinator issue all four Task calls in a single response.  C. Start four separate coordinator processes, one per subtopic.  D. Move the subagents to a smaller model.

<details><summary>Answer</summary>

**B.** Parallel subagents come from multiple Task (Agent) calls in one response. A changes nothing about concurrency. C loses the single coordinator that aggregates and checks coverage. D shortens each run but keeps them serial; it treats the symptom.
</details>

**Q3 (1.2).** The web research subagent needs a figure the document subagent found earlier. A developer proposes letting subagents message each other directly. Which approach fits the coordinator-subagent pattern?
A. Give every subagent every tool so none depends on another.  B. Store all agents' messages in one shared conversation history.  C. Let subagents call each other peer-to-peer when needed.  D. Route it through the coordinator, which includes the earlier finding (with its source) in the web subagent's next task prompt.

<details><summary>Answer</summary>

**D.** Hub and spoke: the coordinator owns information flow, so it stays observable and consistent. A breaks least privilege. B destroys context isolation and bloats every agent's context. C creates untraceable flows and duplicated work.
</details>

**Q4 (1.3).** A document-analysis subagent sometimes fetches web pages and edits local files, which it should never do. Its AgentDefinition has a description and prompt but no `tools` field. Best fix?
A. Set `tools` to only the document tools it needs.  B. Add "never use web or file tools" to its prompt.  C. Remove Task from the coordinator's allowed tools.  D. Lower the subagent's `maxTurns`.

<details><summary>Answer</summary>

**A.** Omitting `tools` means it inherits every available tool. Restricting the toolset makes the bad action impossible rather than discouraged (deterministic over probabilistic). B is guidance with a non-zero failure rate. C stops all delegation. D does not address capability.
</details>

**Q5 (1.2).** After synthesis, the coordinator's report lists "no evidence collected on delivery performance" under gaps. What should the coordinator do next?
A. Publish the report; the gap is disclosed.  B. Delegate a focused follow-up task on delivery performance, passing what is already known, then re-synthesize.  C. Re-run the whole pipeline with twice as many subagents.  D. Ask the synthesis agent to fill the gap from general knowledge.

<details><summary>Answer</summary>

**B.** Iterative refinement: check the synthesis for gaps, re-delegate narrowly, re-synthesize. A is acceptable only when the gap cannot be closed. C is disproportionate. D fabricates evidence.
</details>

In [ ]:
MY_ANSWERS = {"Q1": "", "Q2": "", "Q3": "", "Q4": "", "Q5": ""}
from answer_keys_s5 import EXAM_ANSWERS
print(f"{sum(MY_ANSWERS[q] == a for q, a in EXAM_ANSWERS.items())}/5   key: {EXAM_ANSWERS}")

## Save your work

The cell below writes your TODO 1 to 5 into `multi_agent.py` **only if they pass the offline checks** when re-imported on their own. Otherwise it installs the reference so Session 14 starts from a working pipeline, and tells you which one you got.

In [ ]:
import importlib

def _save_multi_agent():
    header = ('"""Session 5: your multi-agent pipeline (TODO 1-5), written by the notebook."""\n'
              "import json\nfrom concurrent.futures import ThreadPoolExecutor\n\n"
              "from orchestration import failed_report, parse_report\n"
              "from research_tools import RESEARCH_TOOL_DEFS, run_research_tool\n"
              "from subagents import REPORT_CONTRACT, SUBAGENTS\n\n\n")
    try:
        body = "\n\n".join(inspect.getsource(f) for f in
                           (build_subagent_prompt, run_subagent, make_delegate_tool, execute_delegations))
        Path("multi_agent.py").write_text(header + body + f"\n\nCOORDINATOR_SYSTEM = {COORDINATOR_SYSTEM!r}\n",
                                          encoding="utf-8")
        mod = importlib.reload(importlib.import_module("multi_agent"))
        passed, total = checks.report(vars(mod), sdk=False)
        if passed == total:
            return "yours"
    except Exception as e:
        print("Could not save your version:", type(e).__name__, e)
    shutil.copy("solutions/multi_agent_ref.py", "multi_agent.py")
    return "reference"

print("\nmulti_agent.py now holds:", _save_multi_agent())
print("sdk_config.py now holds:", _write_sdk_config())

## Homework (from the curriculum)

1. **Read** the Agent SDK docs on subagents (definition, what subagents inherit, detecting invocations, tool restrictions).
2. **Exercise 4, steps 1 and 2**
   * Step 1: run `s5_sdk_research.py`: a coordinator with at least 2 subagents, `Agent`/`Task` in its allowed tools, and each subagent receiving its findings **in its prompt**, not by inheritance. Save `sdk_report_parallel.md`.
   * Step 2: run it again with `--sequential`. Fill in the table below from the two summary lines and write two sentences on why the gap is the size it is.

| Run | Wall time (s) | Subagents spawned | Max concurrent | Cost ($) |
|---|---|---|---|---|
| parallel | | | | |
| sequential | | | | |

3. **Extension (optional):** add a fourth subagent, `support_analyst`, with one new tool over `NW-SUP-Q3` only, without touching the coordinator code. Run `python s5_sdk_research.py --with-support` once you have written it (the flag imports `solutions/support_analyst_ext.py`; replace it with your own module). Does it get used? Why (or why not)? What does that tell you about descriptions?

**Next session (6):** back to Exercise 1. Hooks that *guarantee* what prompts only request: the $500 refund block, prerequisite gates, structured handoffs and escalation.